In [1]:
%load_ext autoreload
%autoreload 2
from logsProcessing_export import display_round_stripe, round_list_tpl, Log

Loading logs from logs...
2504 files found.
DONE. Loaded 2504 completed game logs.


In [2]:
def create_click_DS(input_ds):
    counter = 5
    dict_clic_tpls = {}
    for round_tuple in input_ds:
        if counter > 0:
            round_data = round_tuple[0]
            game_id = round_tuple[1]

            round_name = "{}_{}".format(game_id, round_data.round_nr)
            
            dial_string = ""
            click_counter = 0
            
            for message in round_data.messages:
                if message.type == "text":
                    dial_string += "{}: {}\n".format(message.speaker, message.text)
        
                if message.type == "selection":
                    click_counter += 1
                    label = "common" if message.text.split()[1] == "<com>" else "different"
                    marking_act = "{} marks image {} as {}\n".format(message.speaker, Log.strip_image_id(message.text.split()[2]), label)

                    dict_clic_tpls[round_name + "_" + str(click_counter)] = (dial_string, marking_act, message.speaker, Log.strip_image_id(message.text.split()[2]), label, round_data.images)
                    dial_string += marking_act

                                            
            counter -= 1
    return dict_clic_tpls
click_DS = create_click_DS(round_list_tpl)

In [3]:
click_DS

{'1048_2_1': ("A: I've got three blue bowls, the one on the left has carrots and celery\nB: I have a picture of square pizza in a dish next to a salad.\nB: I don't have that one.\nB: wait\nA: Is the salad on the right in a clear bowl?\n",
  'A marks image 161846 as different\n',
  'A',
  161846,
  'different',
  {'A': ['bowl_dining_table/COCO_train2014_000000386603.jpg',
    'bowl_dining_table/COCO_train2014_000000144797.jpg',
    'bowl_dining_table/COCO_train2014_000000161846.jpg',
    'bowl_dining_table/COCO_train2014_000000310714.jpg',
    'bowl_dining_table/COCO_train2014_000000244425.jpg',
    'bowl_dining_table/COCO_train2014_000000492731.jpg'],
   'B': ['bowl_dining_table/COCO_train2014_000000386603.jpg',
    'bowl_dining_table/COCO_train2014_000000244425.jpg',
    'bowl_dining_table/COCO_train2014_000000260248.jpg',
    'bowl_dining_table/COCO_train2014_000000086285.jpg',
    'bowl_dining_table/COCO_train2014_000000161846.jpg',
    'bowl_dining_table/COCO_train2014_000000395097

In [4]:
device = "cuda:1"

In [5]:
import torch

In [6]:
import re
from PIL import Image
from llava.model.builder import load_pretrained_model
from llava.mm_utils import process_images, tokenizer_image_token
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
from matplotlib import pyplot as plt

/srv/data/gusloryst/llava3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
model_path = "liuhaotian/llava-v1.5-7b"
tokenizer, model, image_processor, context_len = load_pretrained_model(model_path, None, model_name="llava_v1_5", 
                                                                       # device_map="auto", 
                                                                       device_map=device, 
                                                                       torch_dtype=torch.float16)

/srv/data/gusloryst/llava3.10/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
Loading checkpoint shards:   0%|                                                                                                                              | 0/2 [00:00<?, ?it/s]/srv/data/gusloryst/llava3.10/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storag

In [8]:
# def run_turn(turn):
#     #{Game_Round_Click : [0:dial_string, 1:marking_act, 2:marking_player, 3:marked_image_number, 4:marked_label, 5:Dictionary of images {player:[img_path]}]}
#     joined_list = turn[5]['A'] + turn[5]['B']
#     joined_list_raw = [(re.search(r'_0*(\d+)\.jpg$', path)).group(1) for path in joined_list]
    
#     missing_path = "images/"
#     joined_path_list = [missing_path + a for a in joined_list]
#     images_tensor = process_images(
#             [Image.open(image) for image in joined_path_list],
#             image_processor,
#             model.config
#         ).to(model.device, dtype=torch.float16)
#     img = images_tensor
    
#     text_history = turn[0]
#     marking_act = turn[1]
#     marking_player = turn[2]
#     marked_image_number = turn[3]
#     marked_label = turn[4]


#     text_history_missing = text_history[:-1]
#     text_history_last = text_history[-1:]

#     prompt_for_generation = f"""
#                             You are a helpful language and vision assistant. You see a chat between two people, A and B. 
#                             They are playing a game in which they are seeing set of 6 images from the photobook album. 
#                             Their task is to find out which photos are common for both of them, and which are different. 
#                             They are chatting between eachother. 
#                             First 6 images you received are A's view, and the other 6 are B's view. Each player does not see what the other player sees.

#                             These are names of the images in according order:
#                             {joined_list_raw}
                            
#                             This is the chat tha has happened between two players so far:
                            
#                             CHAT:
#                             {text_history}

#                             Based on the images, their names, and chat between two players, answer following task. Player {marking_player} at this stage of conversation marked one of the images as {marked_label}. Which image do you think was marked?
#                             """

    
#     full_input_ids = tokenizer_image_token(prompt_for_generation, tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt")
#     full_input_ids = full_input_ids.unsqueeze(0).to(model.device)
#     #get the length of the target message
#     targets_as_input_ids = tokenizer_image_token(text_history_last[0], tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt")
#     target_len = targets_as_input_ids.unsqueeze(0).to(model.device).shape[1]
#     # calculate loss, i.e. for perplexity calculation
#     # https://huggingface.co/docs/transformers/v4.37.2/en/perplexity
#     # https://huggingface.co/spaces/evaluate-metric/perplexity
#     targets_as_input_ids = tokenizer_image_token(text_history_last[0], tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt")
#     targets_as_input_ids = targets_as_input_ids.unsqueeze(0).to(model.device)

#     initial_context_len = full_input_ids.shape[-1] - target_len
#     seq_len = target_len
#     nlls = []
#     begin_loc = 0
#     prev_end_loc = initial_context_len
#     stride = 1
#     for end_loc in range(initial_context_len, initial_context_len + seq_len, stride):
#         input_ids = full_input_ids[:, begin_loc:end_loc].to(model.device)
#         target_ids = input_ids.clone()
#         target_ids[:, :-1] = -100
#         with torch.no_grad():
#             outputs = model(input_ids, images=img, image_sizes=[(336, 336)], labels=target_ids)
#             loss = outputs.loss
#             neg_log_likelihood = loss
#         nlls.append(neg_log_likelihood)
#         prev_end_loc = end_loc
#         stride += 1
#         if end_loc == initial_context_len:
#             break
#     ppl = torch.exp(torch.stack(nlls).mean())
    
#     # print(joined_list_raw)
#     # print(text_history)
#     # print(text_history_missing)
#     # print(text_history_last)
#     # print(images_tensor)

In [33]:
def run_turn(turn, max_new_tokens=64, do_sample=False, temperature=0.2):
    joined_list = turn[5]["A"] + turn[5]["B"]
    joined_list_raw = [re.search(r"_0*(\d+)\.jpg$", path).group(1) for path in joined_list]

    missing_path = "images/"
    joined_path_list = [missing_path + a for a in joined_list]

    images = [Image.open(path).convert("RGB") for path in joined_path_list]
    image_sizes = [img.size for img in images]

    images_tensor = process_images(
        images,
        image_processor,
        model.config
    ).to(model.device, dtype=torch.float16)

    # if isinstance(images_tensor, list):
    #     print("LIST")
    #     images_tensor = [x.to(model.device, dtype=torch.float16) for x in images_tensor]
    # else:
    #     print("NO LIST")
    #     images_tensor = images_tensor.to(model.device, dtype=torch.float16)

    text_history = turn[0]
    marking_act = turn[1]
    marking_player = turn[2]
    marked_image_number = turn[3]
    marked_label = turn[4]

    prompt_for_generation = f"""
You are a helpful language and vision assistant. You see a chat between two people, A and B.
They are playing a game in which they are seeing a set of 6 images from the photobook album.
Their task is to find out which photos are common for both of them, and which are different.
They are chatting between each other.
First 6 images you received are A's view, and the other 6 are B's view. Each player does not see what the other player sees.

These are names of the images in according order:
{joined_list_raw}

This is the chat that has happened between two players so far:

CHAT:
{text_history}

Based on the images, their names, and chat between two players, answer the following task:
Player {marking_player} at this stage of conversation marked one of the images as {marked_label}.
Which image do you think was marked?
"""

    input_ids = tokenizer_image_token(
        prompt_for_generation,
        tokenizer,
        IMAGE_TOKEN_INDEX,
        return_tensors="pt"
    ).unsqueeze(0).to(model.device)

    # print("IMAGE TENSOR DATA:")
    # print(images_tensor.shape)
    # print(images_tensor.dtype)
    # print(images_tensor.device)

    
    with torch.inference_mode():
        output_ids = model.generate(
            input_ids,
            images=images_tensor,
            image_sizes=image_sizes,
            do_sample=do_sample,
            temperature=temperature,
            max_new_tokens=max_new_tokens,
            use_cache=True,
        )

    if output_ids.shape[1] > input_ids.shape[1]:
        new_tokens = output_ids[0, input_ids.shape[1]:]
    else:
        new_tokens = output_ids[0]

    # print("PROMPT")
    # print(prompt_for_generation)
    print("GIVEN IMAGES")
    print(joined_list)
    print("CORRECT")
    print(marked_image_number)
    print("ANSWER")

    
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return answer
    # print(answer)

In [34]:
for key in click_DS.keys():
    print("KEY:")
    print(key)
    print(run_turn(click_DS[key]))
    print("\n")

KEY:
1048_2_1
GIVEN IMAGES
['bowl_dining_table/COCO_train2014_000000386603.jpg', 'bowl_dining_table/COCO_train2014_000000144797.jpg', 'bowl_dining_table/COCO_train2014_000000161846.jpg', 'bowl_dining_table/COCO_train2014_000000310714.jpg', 'bowl_dining_table/COCO_train2014_000000244425.jpg', 'bowl_dining_table/COCO_train2014_000000492731.jpg', 'bowl_dining_table/COCO_train2014_000000386603.jpg', 'bowl_dining_table/COCO_train2014_000000244425.jpg', 'bowl_dining_table/COCO_train2014_000000260248.jpg', 'bowl_dining_table/COCO_train2014_000000086285.jpg', 'bowl_dining_table/COCO_train2014_000000161846.jpg', 'bowl_dining_table/COCO_train2014_000000395097.jpg']
CORRECT
161846
ANSWER
Note: You can only see the images and the chat, you don't have any additional information about the game.


KEY:
1048_2_2
GIVEN IMAGES
['bowl_dining_table/COCO_train2014_000000386603.jpg', 'bowl_dining_table/COCO_train2014_000000144797.jpg', 'bowl_dining_table/COCO_train2014_000000161846.jpg', 'bowl_dining_table/